# Loading data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.cluster import KMeans

In [ ]:
X = np.load("data/p1/X.npy")
y = np.load("data/p1/y.npy")

In [ ]:
# cells x genes
X.shape

In [ ]:
# number of labels
y.shape

In [ ]:
# largest entry in the first column
np.max(X[:,0])

# Transform data

In [ ]:
log_X = np.log2(X + 1)

In [ ]:
#largest entry in the first column of processed data
np.max(log_X[:,0])

# Principal Components and Explained Variance

In [ ]:
pca = PCA().fit(X)
log_pca = PCA().fit(log_X)
# fit() is used instead of fit_transform() bc we want to preserve original data for 
# analyzing proportion of variance explained by PCA

In [ ]:
# percentage of the variance explained by the first principal component in raw data
pca.explained_variance_ratio_[0]

In [ ]:
# percentage of the variance explained by the first principal component in transformed data
log_pca.explained_variance_ratio_[0]

In [ ]:
# Cumulative variance explained plots
plt.plot(range(1, X.shape[0]+1), np.cumsum(pca.explained_variance_ratio_), color="black", label="raw data")
plt.plot(range(1, X.shape[0]+1), np.cumsum(log_pca.explained_variance_ratio_), color="pink", label="transformed data")
plt.legend()
plt.show()

In [ ]:
# how many PC's are needed to explain 85% of the variance for raw data
np.where(np.cumsum(pca.explained_variance_ratio_) >= .85)[0][0] + 1

In [ ]:
# how many PC's are needed to explain 85% of the variance for transformed data
np.where(np.cumsum(log_pca.explained_variance_ratio_) >= .85)[0][0] + 1

# PCA scatterplot
used top 2 PCs

In [ ]:
# z contains the data transformed into the 
# new principal component coordinate system
z = pca.fit_transform(log_X)

In [ ]:
plt.scatter(z[:,0],z[:,1],c=y)
plt.title("Scatter plot of 1st and 2nd PCs",size=18)
plt.xlabel("PC 1",size=14)
plt.ylabel("PC 2",size=14)
plt.axis("equal")
plt.show()

found 3 visually distinct clusters.

# MDS

In [ ]:
mds=MDS(n_components=2).fit_transform(log_X)
plt.scatter(mds[:,0], mds[:,1], c=y)

found 3 distinct clusters.

# T-SNE
Project the data onto the top 50 PC's and run T-SNE with a perplexity value of 40 on the projected data

perplexity is a hyperparameter that sets the number of effective nearest neighbors a data point considers

In [ ]:
z_tsne = TSNE(n_components=2,perplexity=40).fit_transform(z[:,0:50])
plt.scatter(z_tsne[:,0],z_tsne[:,1], c=y)

# Visualizing K-means Clustering
continue to use the log-transformed data projected onto the top 50 PC's

In [ ]:
# PCA plot
kmeans = KMeans(5, random_state=0, n_init=50, tol=1e-6)
kmeans.fit(z[:,0:50])
plt.scatter(z[:,0],z[:,1], c=kmeans.labels_)

In [ ]:
# MDS
plt.scatter(mds[:,0],mds[:,1],c=kmeans.labels_)

In [ ]:
# T-SNE
plt.scatter(z_tsne[:,0],z_tsne[:,1], c=kmeans.labels_)

# Elbow Method

In [ ]:
all_kmeans = [i for i in range(8)]
for i in range(8):
    cur_kmeans = KMeans(i+2)
    cur_kmeans.fit(z[:,0:50])
    print("Num clusters", i+2, "Inertia:", cur_kmeans.inertia_)
    all_kmeans[i] = cur_kmeans
plt.plot([i+2 for i in range(8)], [all_kmeans[i].inertia_ for i in range(8)])

the elbow plot changes each time due to random initial centroids chosen by sklearn. take K = 4

# Visualizing cluster means
cluster means: The average gene-expression profile of all cells in that cluster.

In [ ]:
cmeans = np.zeros((5,log_X.shape[1]))
for c in range(5):
    cmeans[c] = np.mean(log_X[np.where(kmeans.labels_==c)[0]],axis=0)

In [ ]:
# MDS on cluster means
mds = MDS(n_components=2,verbose=1,eps=1e-5)
mds.fit(cmeans)
plt.scatter(mds.embedding_[:,0],mds.embedding_[:,1],c=[0,1,2,3,4],s=100)

In [ ]:
# PCA on cluster means
z_means = PCA(2).fit_transform(cmeans)
plt.scatter(z_means[:,0],z_means[:,1],c=[0,1,2,3,4],s=100)

In [ ]:
# tSNE on cluster means
z_means_tsne = TSNE(n_components=2,perplexity=4).fit_transform(cmeans)
plt.scatter(z_means_tsne[:,0],z_means_tsne[:,1],c=[0,1,2,3,4],s=100)

- The MDS and PCA plots confirm there are 3 groups of clusters, one group with 1 cluster mean and two groups with 2 cluster means each.
- The PCA and MDS plots show relatively accurate representation of distances, with one cluster means far away from the others, indicating a different type of cell.
- The T-SNE does not show any grouping of the cluster means, and place them at similar distances apart